In [ ]:
# Imports
%load_ext autoreload
%autoreload 2

import scipy
from tqdm import tqdm
import numpy as np
import gc
import random
import pickle
from glob import *
import numpy as np
from time import time
from torch import tensor as tt
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from matplotlib.pyplot import *
from IPython.display import clear_output
def t2n(x): return x.squeeze().cpu().detach().numpy()

import seaborn as sns
sns.set(rc={"figure.figsize": (8, 5)})
sns.set_style("darkgrid")

In [ ]:
###############################
# Import the dependencies
###############################
from ReasoningCombinatorials.one_level_guess_approach.board_enumeration import *
from ReasoningCombinatorials.one_level_guess_approach.sudoku_generator import *
from ReasoningCombinatorials.one_level_guess_approach.single_guess import *
from ReasoningCombinatorials.one_level_guess_approach.data_generator import *

In [ ]:
# Print an Example
dataset = SudokuDataset()
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True)

l,X,Y = next(iter(dataloader))
print(l,X,Y)

In [ ]:
###############################
# Import the dependencies
###############################
from ReasoningCombinatorials.one_level_guess_approach.transformer_model import *
from ReasoningCombinatorials.one_level_guess_approach.optimizer import *
from ReasoningCombinatorials.one_level_guess_approach.evaluation import *

In [ ]:
###############################
# Training loop
###############################

# Define loss function for guess optimization
loss_type = 'cross_entropy'   #L2 in paper
# loss_type = 'logsumexp'     #L1 in paper
#====================================

bs_train = 32
dataloader = torch.utils.data.DataLoader(dataset, batch_size=bs_train, shuffle=True)

upper_bound_list, lower_bound_list, inv_probs_list = [], [], []
losses = []
cell_accs = []
best_loss = 9999
for i in tqdm(range(total_steps)):
    model.train()
    torch.cuda.empty_cache()

    l,X,Y = next(iter(dataloader))
    X,Y = X.to(device), Y.bool().to(device)

    #===================================
    opt.zero_grad()

    out = model(X)
    z = torch.log_softmax(out, dim=-1)
    probs = torch.softmax(out, dim=-1)

    #===========================================================================
    #  Rule Application Loss | Cross-Entropy Loss up to token 101 (multi-target)
    #===========================================================================
    sum_log_softmax_before101 = 0
    pos_101 = torch.where(X == 101)[1]  # [bs]
    for j in range(X.shape[0]):
        k = pos_101[j].item()  # index of token 101

        for t in range(k):  # loop through tokens before 101
            log_probs_t = z[j][t]       # [vocab]
            targets_t = Y[j][t]         # [vocab], multi-hot

            sum_log_softmax_before101 += log_probs_t[targets_t].sum()

    loss0 = - sum_log_softmax_before101 / X.shape[0]

    #==================
    # Scratchpad Loss
    #==================
    mask = (X == 103)
    target_indices = torch.argmax(Y.float(), dim=-1)
    target_indices_int64 = target_indices.long()
    target_log_probs = torch.gather(z, dim=-1, index=target_indices_int64.unsqueeze(-1)).squeeze(-1)
    masked_log_probs = target_log_probs * mask
    sum_log_softmax = masked_log_probs.sum()

    loss1 = - sum_log_softmax / X.shape[0]

    #=====================================
    # Guess Loss (Backdoors multi-targets)
    #=====================================
    if loss_type == 'cross_entropy':
        sum_log_softmax101 = 0
        pos_101 = torch.where(X == 101)[1]
        for j in range(X.shape[0]):
            k = pos_101[j]
            log_softmax_101_k = z[j][k]
            targets_101_k = Y[j][k]

            sum_log_softmax101 += log_softmax_101_k[targets_101_k].sum()

        loss2 = - sum_log_softmax101 / X.shape[0]
    #=========================================
    if loss_type == 'logsumexp':
        sum_logsumexp = 0
        pos_101 = torch.where(X == 101)[1]
        for j in range(X.shape[0]):
            k = pos_101[j]
            out_101 = out[j][k]
            targets_101 = Y[j][k]

            nomin = torch.logsumexp(out_101[targets_101],dim=-1)
            denom = torch.logsumexp(out_101,dim=-1)
            if nomin.isinf(): continue # in some rare cases (0.2%), there are no backdoors (no targets to guess for this instance)
            sum_logsumexp += nomin - denom

        loss2 = - sum_logsumexp / X.shape[0]
    #=========================================

    loss = loss0 + loss1 + loss2

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    #===================================
    if i%20==0:
        clear_output(wait=True)

        losses.append(loss.item())
        recent_loss = np.mean(losses[-30:])
        plot(losses), title(f'{recent_loss:2.3f}'),
        show()
        if recent_loss < best_loss:
            best_loss = recent_loss
            torch.save(model.state_dict(),'best_model.pt')

        acci = get_cell_acc(X,Y,z)
        cell_accs.append(acci)
        recent_cellacc = np.mean(cell_accs[-30:])
        plot(cell_accs), title(f'{recent_cellacc:2.3f}'),
        show()

        update_upper_bound(X, Y, upper_bound_list)
        update_lower_bound(X, Y, lower_bound_list)
        update_inv_prob(X, Y, probs, inv_probs_list)
        Nhist = 5000
        cumulative_histogram(upper_bound_list[-Nhist:], lower_bound_list[-Nhist:] ,inv_probs_list[-Nhist:], 95)